# Preprocesamiento de datos

En este notebook se realiza el preprocesamiento del dataset con el objetivo de preparar los datos para el entrenamiento de los diferentes modelos.

Las principales tareas que se realizarán en esta fase son:

- Limpieza de datos
- Tratamiento de valores faltantes
- Codificación de variables categóricas
- Transformación de variables
- Preparación del dataset final para el modelado

Para ello se creo el scrip `preprocessing.py`este se encarga de hacer una limpieza general. La idea es ir mirando si lo hace de manera correcta ajustando las funciones para que al final se puedan usar estas de manera definitiva para tener un mejor pipeline. Al principio se fue haciendo prueba error para ver si realmente estaba limpiando y procesando bien los ficheros -> esto es un fase de varios intentos 

Debido al gran volumen del dataset MIMIC-IV, se seleccionaron únicamente las tablas más relevantes, concretamente patients, admissions y diagnoses.

Esto se hizo ya que el ordenador desde donde se trabajo no presentaba suficiente RAM para procesar todo los datos y hacer merge de las tablas

En una primera fase se construyó un dataset base a partir de las tablas patients y admissions, que contienen información demográfica del paciente y datos sobre cada ingreso hospitalario.

Posteriormente se generó la variable objetivo de readmisión hospitalaria a partir de la secuencia temporal de admisiones de cada paciente. 



El proceso de preprocesamiento se dividió en dos etapas principales.

En la primera etapa se realizó una limpieza estructural del dataset, incluyendo la eliminación de duplicados, conversión de variables temporales y generación de variables derivadas como la duración de un paciente en el hospital. Además, se integraron las tablas de pacientes y admisiones en una sola tabla -> Para ellos se definio la función `run_preprocessing_part1` 

En una segunda etapa se abordó el tratamiento de valores faltantes, la codificación de variables categóricas y la preparación final del dataset para su uso en modelos de ML. Para ellos se definio la función `run_preprocessing_part2` 


NOTA: falta por terminar la segunda parte ahora mismo no funciona del todo  la función -> FUNCIONA

# 1 PARTE

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from src.data.preprocessing import run_preprocessing

df = run_preprocessing_part1()

df.head()

Loading datasets...
Loading patients...
Loading admissions...
Loading diagnoses...
Cleaning datasets...
Merging datasets...
Saving interim dataset...
Preprocessing completed.


,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,...,race,edregtime,edouttime,hospital_expire_flag,length_of_stay,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,...,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0,0,F,52,2180,2014 - 2016,2180-09-09
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,...,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0,1,F,52,2180,2014 - 2016,2180-09-09
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,...,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0,1,F,52,2180,2014 - 2016,2180-09-09
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,...,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0,2,F,52,2180,2014 - 2016,2180-09-09
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,...,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0,0,F,19,2160,2008 - 2010,NaN


In [22]:
df.shape


(546028, 22)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int64         
 1   hadm_id               546028 non-null  int64         
 2   admittime             546028 non-null  datetime64[ns]
 3   dischtime             546028 non-null  datetime64[ns]
 4   deathtime             11790 non-null   object        
 5   admission_type        546028 non-null  object        
 6   admit_provider_id     546024 non-null  object        
 7   admission_location    546027 non-null  object        
 8   discharge_location    396210 non-null  object        
 9   insurance             536673 non-null  object        
 10  language              545253 non-null  object        
 11  marital_status        532409 non-null  object        
 12  race                  546028 non-null  object        
 13 

In [4]:
df.head

<bound method NDFrame.head of         subject_id   hadm_id           admittime           dischtime  \
0         10000032  22595853 2180-05-06 22:23:00 2180-05-07 17:15:00   
1         10000032  22841357 2180-06-26 18:27:00 2180-06-27 18:49:00   
2         10000032  25742920 2180-08-05 23:44:00 2180-08-07 17:50:00   
3         10000032  29079034 2180-07-23 12:35:00 2180-07-25 17:55:00   
4         10000068  25022803 2160-03-03 23:16:00 2160-03-04 06:26:00   
...            ...       ...                 ...                 ...   
546023    19999828  25744818 2149-01-08 16:44:00 2149-01-18 17:00:00   
546024    19999828  29734428 2147-07-18 16:23:00 2147-08-04 18:10:00   
546025    19999840  21033226 2164-09-10 13:47:00 2164-09-17 13:42:00   
546026    19999840  26071774 2164-07-25 00:27:00 2164-07-28 12:15:00   
546027    19999987  23865745 2145-11-02 21:38:00 2145-11-11 12:57:00   

                  deathtime  admission_type admit_provider_id  \
0                       NaN          URG

In [24]:
df.isnull().sum().sort_values(ascending=False)

deathtime               534238
dod                     401062
edregtime               166788
edouttime               166788
discharge_location      149818
marital_status           13619
insurance                 9355
language                   775
admit_provider_id            4
admission_location           1
subject_id                   0
dischtime                    0
admittime                    0
hadm_id                      0
admission_type               0
race                         0
length_of_stay               0
hospital_expire_flag         0
gender                       0
anchor_age                   0
anchor_year                  0
anchor_year_group            0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df["length_of_stay"].describe()

count    546028.000000
mean          4.221033
std           7.201291
min          -1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max         515.000000
Name: length_of_stay, dtype: float64

In [8]:
df["subject_id"].nunique()

223452

In [25]:
import os
os.listdir("../data/interim")

['.gitkeep', 'cleaned_dataset.csv']

In [10]:
summary = {
    "n_rows": df.shape[0],
    "n_columns": df.shape[1],
    "n_patients": df["subject_id"].nunique(),
    "n_admissions": df["hadm_id"].nunique()
}

summary

{'n_rows': 546028,
 'n_columns': 22,
 'n_patients': 223452,
 'n_admissions': 546028}

# 2 PARTE DEL PREPROCESADO

En esta parte se utiliza directamente el dataset limpio `cleaned_dataset` para eliminar columnas que se consideran irrelevantes, valores nulo, codificar variables categóricas para preparar el conjunto final para entrenarlo con modelos.

 Partir de este mejora pretende mejorar el pipeline ya que permite reutilizar los datos ya limpiados sin necesidad de repetir las operaciones iniciales de procesamiento.

In [ ]:
import pandas as pd
from src.config import DATA_INTERIM
df = pd.read_csv(DATA_INTERIM / "cleaned_dataset.csv")
# PROCESO MANUAL DE TODA LA VIDA 
df.head()
df.isnull().sum().sort_values(ascending=False)

deathtime               534238
dod                     401062
edregtime               166788
edouttime               166788
discharge_location      149818
marital_status           13619
insurance                 9355
language                   775
admit_provider_id            4
admission_location           1
subject_id                   0
dischtime                    0
admittime                    0
hadm_id                      0
admission_type               0
race                         0
length_of_stay               0
hospital_expire_flag         0
gender                       0
anchor_age                   0
anchor_year                  0
anchor_year_group            0
dtype: int64

# Eliminamos variables irrelevantes e imputamos variables categóricas


In [34]:
#Eliminamos la variable de deathtime, ya que no es relevante para el objetivo de predecir readmisiones y podría introducir sesgo y ya tenemos la variable 
#hospital_expire_flag que indica si el paciente falleció durante la hospitalización.
df = df.drop(columns=["deathtime"])

#Tambien eliminar la variable dod que se indica si el paciente falleció fuera del hospital, ya que no
#  es relevante para el objetivo de predecir readmisiones 

df = df.drop(columns=["dod"])

#También eliminamos las variables relacionadas con Urgencias.
df = df.drop(columns=["edregtime", "edouttime"])
#No se usan directamente en modelos porque ya se creo length_of_stay
df = df.drop(columns=["admittime", "dischtime"])
#Tampoco aporta informacón relevante 
df = df.drop(columns=["anchor_year_group"])

#Imputamos las variables categóricas con "Uknown" -> No se si es la solución mas adecuada ?¿?¿?¿ porque asi el modelo entiende que se desconoce esa información, pero no es lo mismo que imputar con la moda o con otro valor.
df["discharge_location"] = df["discharge_location"].fillna("Unknown")
df["marital_status"] = df["marital_status"].fillna("Unknown")
df["insurance"] = df["insurance"].fillna("Unknown")
df["language"] = df["language"].fillna("Unknown")

df = df.drop(columns=["admit_provider_id"]) #identificador del médico que admitió al paciente -> irrelevante

df["admission_location"] = df["admission_location"].fillna("Unknown")

Algunas de estas variables, como deathtime o edregtime, fueron eliminadas al no aportar información relevante para el problema de predicción de readmisión o por corresponder a situaciones específicas como ingresos a través del servicio de urgencias.

En el caso de variables categóricas con un número moderado de valores faltantes, como insurance, marital_status o language, se puso "Unknown".

In [35]:
df.isnull().sum().sort_values(ascending=False)

subject_id              0
hadm_id                 0
admission_type          0
admission_location      0
discharge_location      0
insurance               0
language                0
marital_status          0
race                    0
hospital_expire_flag    0
length_of_stay          0
gender                  0
anchor_age              0
anchor_year             0
dtype: int64

In [36]:
df.select_dtypes(include="object").columns

Index(['admission_type', 'admission_location', 'discharge_location',
       'insurance', 'language', 'marital_status', 'race', 'gender'],
      dtype='object')

 Se codifica las variables de tipo objeto en 0 1

In [37]:
categorical_cols = [
    "gender",
    "race",
    "insurance",
    "marital_status",
    "language",
    "admission_type",
    "admission_location",
    "discharge_location"
]

df = pd.get_dummies(df, columns=categorical_cols)

In [38]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Columns: 113 entries, subject_id to discharge_location_Unknown
dtypes: bool(107), int64(6)
memory usage: 80.7 MB


In [39]:
df.head()

,subject_id,hadm_id,hospital_expire_flag,length_of_stay,anchor_age,anchor_year,gender_F,gender_M,race_AMERICAN INDIAN/ALASKA NATIVE,race_ASIAN,...,discharge_location_DIED,discharge_location_HEALTHCARE FACILITY,discharge_location_HOME,discharge_location_HOME HEALTH CARE,discharge_location_HOSPICE,discharge_location_OTHER FACILITY,discharge_location_PSYCH FACILITY,discharge_location_REHAB,discharge_location_SKILLED NURSING FACILITY,discharge_location_Unknown
0,10000032,22595853,0,0,52,2180,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
1,10000032,22841357,0,1,52,2180,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
2,10000032,25742920,0,1,52,2180,True,False,False,False,...,False,False,False,False,True,False,False,False,False,False
3,10000032,29079034,0,2,52,2180,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
4,10000068,25022803,0,0,19,2160,True,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [40]:
df.select_dtypes(include="object")
len(df.columns)


113

In [41]:
#Finalmente se eliminan los identificadores de paciente y hospitalización.

df = df.drop(columns=["subject_id", "hadm_id"])
len(df.columns)

111

AHORA LA IDEA ES HACER LO ANTERIOR METIDO EN LA FUNCION run_preprocessing_part2

In [ ]:
import sys
import os
import pandas as pd 
sys.path.append(os.path.abspath(".."))
from src.config import DATA_INTERIM
from src.data.preprocessing import run_preprocessing_part2

df =pd.read_csv(DATA_INTERIM/"cleaned_dataset.csv",
     parse_dates=["admittime","dischtime"]
     )
df.columns

Index(['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag', 'length_of_stay',
       'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod'],
      dtype='object')

In [ ]:
df_model = run_preprocessing_part2(df)

Preparing dataset for modeling...
Creating readmission target variable...
Columns before processing: 24
Columns after encoding: 97
Saving processed dataset...
Model dataset created.


In [19]:
df_model[["previous_admissions","readmission_30_days"]].head(10)

,previous_admissions,readmission_30_days
0,0,0
1,1,1
3,2,1
5,0,0
8,0,0
15,0,0
19,0,0
16,1,0
17,2,0
22,0,1


#Ahora ya funciona igual la funcion que hacer el preprocesado en el notebook que la función que hace el preprocesado en el script, lo que me permite tener un control total sobre el proceso de preprocesado y poder hacer cambios fácilmente sin tener que modificar el código del script. Además, me permite tener un notebook limpio y organizado, donde solo se muestra el resultado final del preprocesado, sin tener que mostrar todo el código intermedio.

In [ ]:
df_model["readmission_30_days"].value_counts()
df_model.head()

,length_of_stay,anchor_age,readmission_30_days,gender_F,gender_M,race_AMERICAN INDIAN/ALASKA NATIVE,race_ASIAN,race_ASIAN - ASIAN INDIAN,race_ASIAN - CHINESE,race_ASIAN - KOREAN,...,discharge_location_DIED,discharge_location_HEALTHCARE FACILITY,discharge_location_HOME,discharge_location_HOME HEALTH CARE,discharge_location_HOSPICE,discharge_location_OTHER FACILITY,discharge_location_PSYCH FACILITY,discharge_location_REHAB,discharge_location_SKILLED NURSING FACILITY,discharge_location_Unknown
0,0,52,0,True,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
1,1,52,1,True,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
3,2,52,1,True,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
2,1,52,0,True,False,False,False,False,False,False,...,False,False,False,False,True,False,False,False,False,False
4,0,19,0,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [ ]:
len(df_model.columns)

111